In [ ]:
def main(datasources, start_date, end_date):
    """BigAlpha 端到端多源 PatchTST 融合优化 v7 conservative。

    代码格式对齐官方 Transformer.ipynb：
        1. 平台只替换 datasources / start_date / end_date。
        2. 训练区间写死，不使用传入的测试区间训练，避免数据泄漏。
        3. 训练读 bigalpha_2026_* 开发表；推理优先读平台注入 datasources。
        4. 最终返回且仅返回 ['date', 'instrument', 'score']。

    本地机器没有 BigQuant DAI 时无法实际取数；平台上 optional 表字段若不匹配会自动跳过。
    v7 回到 v5 主干，只加入更保守的微观结构残差候选和窄权重扰动。
    """
    import math
    import random
    import time

    import numpy as np
    import pandas as pd
    import dai
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    try:
        import structlog
        logger = structlog.get_logger()
    except Exception:
        class _Logger:
            def info(self, *args, **kwargs):
                print("INFO", args, kwargs)

            def warning(self, *args, **kwargs):
                print("WARN", args, kwargs)

        logger = _Logger()

    # ---------- 固定配置 ----------
    SEED = 2026
    EPS = 1e-8
    TRAIN_START = "2021-01-01 00:00:00"
    TRAIN_END = "2023-06-30 23:59:59"
    VALID_START = "2023-07-01 00:00:00"
    VALID_END = "2023-12-31 23:59:59"

    SEQ_LEN = 60
    PATCH_LEN = 12
    STRIDE = 6
    EPOCHS = 4
    BATCH = 512
    LR = 4e-4
    MAX_TRAIN_SAMPLES = 50000
    MAX_VALID_SAMPLES = 12000
    MAX_FEATURES = 90

    DEV_TABLES = {
        "instruments": "bigalpha_2026_instruments",
        "bar1d": "bigalpha_2026_bar1d",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar15m": "bigalpha_2026_stock_bar15m",
        "financial": "bigalpha_2026_financial",
        "factorlib": "bigalpha_2026_factorlib",
        "exposure": "bigalpha_2026_exposure",
    }

    def ds(key):
        """兼容平台 datasources 的不同命名；没有注入时使用开发表名。"""
        aliases = {
            "instruments": ["instruments", "instrument", "stock_pool", DEV_TABLES["instruments"]],
            "bar1d": ["bar1d", "stock_bar1d", DEV_TABLES["bar1d"]],
            "bar5m": ["bar5m", "stock_bar5m", DEV_TABLES["bar5m"]],
            "bar15m": ["bar15m", "stock_bar15m", DEV_TABLES["bar15m"]],
            "financial": ["financial", "finance", DEV_TABLES["financial"]],
            "factorlib": ["factorlib", "factor_library", DEV_TABLES["factorlib"]],
            "exposure": ["exposure", "style", DEV_TABLES["exposure"]],
        }
        if isinstance(datasources, dict):
            for name in aliases[key]:
                if name in datasources:
                    return datasources[name]
        return DEV_TABLES[key]

    PRED_TABLES = {k: ds(k) for k in DEV_TABLES}

    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    def fmt_start(x):
        return pd.to_datetime(x).strftime("%Y-%m-%d 00:00:00")

    def fmt_end(x):
        return pd.to_datetime(x).strftime("%Y-%m-%d 23:59:59")

    def norm_day(x):
        return pd.to_datetime(x).normalize()

    def safe_query(sql, filters, name, required=True):
        try:
            return dai.query(sql, filters=filters, compression=True).df()
        except Exception as exc:
            if required:
                raise
            logger.warning("可选数据读取失败，已跳过", name=name, error=str(exc))
            return pd.DataFrame()

    def clean_keys(df):
        if df.empty:
            return df
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"]).dt.normalize()
        df["instrument"] = df["instrument"].astype(str)
        return df

    def to_num(df, cols):
        for c in cols:
            if c in df.columns:
                df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
        return df

    def cs_clean(s):
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
        if s.notna().sum() < 3:
            return pd.Series(0.0, index=s.index)
        lo, hi = s.quantile([0.01, 0.99])
        if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
            s = s.clip(lo, hi)
        med = s.median()
        s = s.fillna(med if np.isfinite(med) else 0.0)
        std = s.std(ddof=0)
        if not np.isfinite(std) or std <= EPS:
            return pd.Series(0.0, index=s.index)
        return (s - s.mean()) / std

    def final_score(df):
        out = df[["date", "instrument", "score"]].copy()
        out["date"] = pd.to_datetime(out["date"]).dt.normalize()
        out["instrument"] = out["instrument"].astype(str)
        out["score"] = pd.to_numeric(out["score"], errors="coerce").replace([np.inf, -np.inf], np.nan)
        out["score"] = out.groupby("date")["score"].transform(cs_clean)
        out["score"] = out.groupby("date")["score"].transform(
            lambda s: s.fillna(0.0).rank(method="average", pct=True) * 2.0 - 1.0
        )
        out["score"] = out["score"].fillna(0.0)
        out = out.drop_duplicates(["date", "instrument"])
        return out.sort_values(["date", "instrument"]).reset_index(drop=True)

    # ---------- 1. instruments 股票池 ----------
    def read_pool(table, sd, ed):
        sql = f"SELECT date, instrument FROM {table}"
        df = safe_query(sql, {"date": [fmt_start(sd), fmt_end(ed)]}, "instruments", required=False)
        if df.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        return clean_keys(df)[["date", "instrument"]].drop_duplicates(["date", "instrument"])

    # ---------- 2. bar1d 基础日频因子 ----------
    BAR1D_COLS = ["open", "high", "low", "close", "volume", "amount"]
    CORE_FEATURES = [
        "ret_1", "ret_5", "ret_10", "ret_20", "ret_40", "ret_60",
        "intraday_ret", "overnight_gap", "range", "body_ratio", "close_location",
        "upper_shadow", "lower_shadow", "shadow_balance", "volume_log", "amount_log",
        "volatility_5", "volatility_10", "volatility_20",
        "amount_ratio5", "volume_ratio5", "amount_ratio20", "volume_ratio20",
        "close_to_ma20",
    ]

    def read_bar1d(table, sd, ed):
        sql = f"""
        SELECT date, instrument, open, high, low, close, volume, amount
        FROM {table}
        ORDER BY instrument, date
        """
        df = safe_query(sql, {"date": [fmt_start(sd), fmt_end(ed)]}, "bar1d", required=True)
        df = clean_keys(df)
        df = to_num(df, BAR1D_COLS)
        return df.drop_duplicates(["date", "instrument"]).sort_values(["instrument", "date"])

    def make_bar1d_features(bar, with_label):
        df = bar.copy().sort_values(["instrument", "date"])
        g = df.groupby("instrument", group_keys=False)
        prev_close = g["close"].shift(1)
        df["ret_1"] = df["close"] / (prev_close + EPS) - 1.0
        df["ret_5"] = g["close"].pct_change(5)
        df["ret_10"] = g["close"].pct_change(10)
        df["ret_20"] = g["close"].pct_change(20)
        df["ret_40"] = g["close"].pct_change(40)
        df["ret_60"] = g["close"].pct_change(60)
        df["intraday_ret"] = df["close"] / (df["open"] + EPS) - 1.0
        df["overnight_gap"] = df["open"] / (prev_close + EPS) - 1.0
        df["range"] = df["high"] / (df["low"] + EPS) - 1.0
        df["body_ratio"] = (df["close"] - df["open"]).abs() / (df["high"] - df["low"] + EPS)
        df["close_location"] = (df["close"] - df["low"]) / (df["high"] - df["low"] + EPS) - 0.5
        df["upper_shadow"] = (df["high"] - np.maximum(df["open"], df["close"])) / (df["close"].abs() + EPS)
        df["lower_shadow"] = (np.minimum(df["open"], df["close"]) - df["low"]) / (df["close"].abs() + EPS)
        df["shadow_balance"] = df["lower_shadow"] - df["upper_shadow"]
        df["volume_log"] = np.log1p(df["volume"].clip(lower=0))
        df["amount_log"] = np.log1p(df["amount"].clip(lower=0))
        df["volatility_5"] = g["ret_1"].transform(lambda s: s.rolling(5, min_periods=3).std())
        df["volatility_10"] = g["ret_1"].transform(lambda s: s.rolling(10, min_periods=5).std())
        df["volatility_20"] = g["ret_1"].transform(lambda s: s.rolling(20, min_periods=8).std())
        df["amount_ma5"] = g["amount"].transform(lambda s: s.rolling(5, min_periods=3).mean())
        df["volume_ma5"] = g["volume"].transform(lambda s: s.rolling(5, min_periods=3).mean())
        df["amount_ma20"] = g["amount"].transform(lambda s: s.rolling(20, min_periods=8).mean())
        df["volume_ma20"] = g["volume"].transform(lambda s: s.rolling(20, min_periods=8).mean())
        df["amount_ratio5"] = df["amount"] / (df["amount_ma5"] + EPS) - 1.0
        df["volume_ratio5"] = df["volume"] / (df["volume_ma5"] + EPS) - 1.0
        df["amount_ratio20"] = df["amount"] / (df["amount_ma20"] + EPS) - 1.0
        df["volume_ratio20"] = df["volume"] / (df["volume_ma20"] + EPS) - 1.0
        df["close_ma20"] = g["close"].transform(lambda s: s.rolling(20, min_periods=8).mean())
        df["close_to_ma20"] = df["close"] / (df["close_ma20"] + EPS) - 1.0
        if with_label:
            df["label"] = g["close"].shift(-1) / (df["close"] + EPS) - 1.0
        return df

    # ---------- 3. 5m / 15m 聚合日频因子 ----------
    def read_intraday_daily(table, sd, ed, prefix):
        # 盘口字段来自图中“分钟附盘口”数据；若平台表字段不全，会被 safe_query 捕获后跳过。
        sql = f"""
        SELECT
            date::DATE::DATETIME AS date,
            instrument,
            FIRST(open) AS {prefix}_open,
            LAST(close) AS {prefix}_close,
            MAX(high) AS {prefix}_high,
            MIN(low) AS {prefix}_low,
            MAX(volume) AS {prefix}_volume,
            MAX(amount) AS {prefix}_amount,
            LAST(close) / NULLIF(FIRST(open), 0) - 1 AS {prefix}_ret,
            MAX(high) / NULLIF(MIN(low), 0) - 1 AS {prefix}_range,
            AVG((ask_price1 - bid_price1) / NULLIF((ask_price1 + bid_price1) / 2, 0)) AS {prefix}_spread,
            AVG((bid_volume1 - ask_volume1) / NULLIF(bid_volume1 + ask_volume1, 0)) AS {prefix}_imbalance
        FROM {table}
        WHERE close > 0
        GROUP BY date::DATE, instrument
        ORDER BY instrument, date
        """
        df = safe_query(sql, {"date": [fmt_start(sd), fmt_end(ed)]}, prefix, required=False)
        if df.empty:
            return pd.DataFrame(columns=["date", "instrument"])
        df = clean_keys(df)
        feat_cols = [c for c in df.columns if c not in ("date", "instrument")]
        df = to_num(df, feat_cols)
        for c in [f"{prefix}_volume", f"{prefix}_amount"]:
            if c in df.columns:
                df[c + "_log"] = np.log1p(df[c].clip(lower=0))
        return df.drop_duplicates(["date", "instrument"])

    # ---------- 4/5. financial 和 factorlib：固定候选列，失败则跳过 ----------
    FINANCIAL_COLS = [
        "operating_revenue", "total_operating_revenue", "operating_profit",
        "total_profit", "net_profit", "net_profit_to_parent_shareholders",
        "total_assets", "total_current_assets", "total_owner_equity",
        "total_liabilities_and_owner_equity", "working_capital",
        "net_working_capital", "interest_bearing_debt", "net_debt",
        "ebit", "ebitda", "nopat", "fcff", "fcfe",
    ]
    FACTORLIB_COLS = (
        [f"alpha_{i:03d}" for i in range(1, 21)]
        + [f"alpha{i:03d}" for i in range(1, 11)]
        + [f"factor_{i}" for i in range(1, 11)]
    )

    def read_optional_columns(table, sd, ed, prefix, candidate_cols, max_cols):
        # 不动态读取全表字段，避免平台字段数检查报错；先尝试整组读取，失败再逐列读取。
        candidate_cols = list(candidate_cols)[:max_cols]
        sql = f"SELECT date, instrument, {', '.join(candidate_cols)} FROM {table} ORDER BY instrument, date"
        df = safe_query(sql, {"date": [fmt_start(sd), fmt_end(ed)]}, prefix, required=False)
        if df.empty:
            pieces = []
            for col in candidate_cols:
                one = safe_query(
                    f"SELECT date, instrument, {col} FROM {table} ORDER BY instrument, date",
                    {"date": [fmt_start(sd), fmt_end(ed)]},
                    f"{prefix}.{col}",
                    required=False,
                )
                if not one.empty:
                    pieces.append(clean_keys(one))
            if not pieces:
                return pd.DataFrame(columns=["date", "instrument"])
            df = pieces[0]
            for one in pieces[1:]:
                df = df.merge(one, how="outer", on=["date", "instrument"])
        df = clean_keys(df)
        keep = []
        for c in [c for c in df.columns if c not in ("date", "instrument")]:
            s = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            if s.notna().mean() >= 0.02 and s.std(ddof=0) > EPS:
                df[c] = s
                keep.append(c)
        keep = keep[:max_cols]
        if not keep:
            return pd.DataFrame(columns=["date", "instrument"])
        out = df[["date", "instrument"] + keep].copy()
        out = out.groupby(["date", "instrument"], as_index=False)[keep].mean()
        return out.rename(columns={c: f"{prefix}_{c}" for c in keep})

    # ---------- 6. exposure 中性化 ----------
    EXPOSURE_COLS = [
        "SIZE", "BETA", "MOMENTUM", "RESVOL", "BTOP", "EARNYILD", "GROWTH",
        "LEVERAGE", "LIQUIDTY", "SIZENL", "size", "beta", "momentum",
        "resvol", "btop", "earnyild", "growth", "leverage", "liquidty",
    ]

    def read_exposure(table, sd, ed):
        exp = read_optional_columns(table, sd, ed, "exp", EXPOSURE_COLS, 18)
        exp_cols = [c for c in exp.columns if c not in ("date", "instrument")]
        return exp, exp_cols

    def neutralize_score(score_df, exposure_df, exposure_cols):
        if score_df.empty or exposure_df.empty or not exposure_cols:
            return score_df
        df = score_df.merge(exposure_df, how="left", on=["date", "instrument"])
        df["score_neu"] = df["score"]
        for _, idx in df.groupby("date").groups.items():
            g = df.loc[idx]
            y = pd.to_numeric(g["score"], errors="coerce").replace([np.inf, -np.inf], np.nan)
            x = g[exposure_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            if y.notna().sum() < max(50, len(exposure_cols) + 10):
                continue
            x = x.apply(lambda s: s.fillna(s.median()), axis=0).fillna(0.0)
            x = x.apply(cs_clean, axis=0)
            valid = y.notna()
            xv = x.loc[valid].to_numpy(dtype=np.float64)
            yv = y.loc[valid].to_numpy(dtype=np.float64)
            if len(yv) < max(50, len(exposure_cols) + 10):
                continue
            design = np.column_stack([np.ones(len(yv)), xv])
            try:
                beta, *_ = np.linalg.lstsq(design, yv, rcond=None)
                df.loc[y.loc[valid].index, "score_neu"] = yv - design @ beta
            except Exception:
                continue
        return df[["date", "instrument", "score_neu"]].rename(columns={"score_neu": "score"})

    def build_panel(tables, sd, ed, with_label, history_days=300, include_optional=True):
        sd0 = norm_day(sd) - pd.Timedelta(days=history_days)
        ed0 = norm_day(ed) + (pd.Timedelta(days=8) if with_label else pd.Timedelta(days=0))
        t0 = time.time()

        pool = read_pool(tables["instruments"], sd0, ed0)
        bar = read_bar1d(tables["bar1d"], sd0, ed0)
        panel = make_bar1d_features(bar, with_label=with_label)
        if pool.empty:
            logger.warning("instruments 为空，临时使用 bar1d 股票池")
            pool = panel[["date", "instrument"]].drop_duplicates()
        pool = pool.drop_duplicates(["date", "instrument"]).copy()
        pool["in_pool"] = True
        panel = panel.merge(pool, how="left", on=["date", "instrument"])
        panel["in_pool"] = panel["in_pool"].fillna(False)

        if include_optional:
            for key, prefix in [("bar5m", "hf5m"), ("bar15m", "hf15m")]:
                hf = read_intraday_daily(tables[key], sd0, ed0, prefix)
                if not hf.empty:
                    panel = panel.merge(hf, how="left", on=["date", "instrument"])
            fin = read_optional_columns(tables["financial"], sd0, ed0, "fin", FINANCIAL_COLS, 16)
            if not fin.empty:
                panel = panel.merge(fin, how="left", on=["date", "instrument"])
            flib = read_optional_columns(tables["factorlib"], sd0, ed0, "flib", FACTORLIB_COLS, 22)
            if not flib.empty:
                panel = panel.merge(flib, how="left", on=["date", "instrument"])

        panel = panel.sort_values(["instrument", "date"]).reset_index(drop=True)
        fill_cols = [c for c in panel.columns if c.startswith("fin_") or c.startswith("flib_")]
        if fill_cols:
            panel[fill_cols] = panel.groupby("instrument", group_keys=False)[fill_cols].ffill()
        logger.info(
            "panel 构建完成",
            rows=len(panel),
            cols=len(panel.columns),
            start=str(norm_day(sd).date()),
            end=str(norm_day(ed).date()),
            elapsed=round(time.time() - t0, 2),
        )
        return panel, pool[["date", "instrument"]]

    def feature_quality(panel, col):
        mask = (
            (panel["date"] >= norm_day(TRAIN_START))
            & (panel["date"] <= norm_day(TRAIN_END))
            & panel["in_pool"].astype(bool)
        )
        s = pd.to_numeric(panel.loc[mask, col], errors="coerce").replace([np.inf, -np.inf], np.nan)
        if len(s) == 0 or s.notna().mean() < 0.02:
            return 0.0
        std = s.std(ddof=0)
        return float(s.notna().mean() * min(std if np.isfinite(std) else 0.0, 10.0))

    def choose_features(panel):
        exclude = {
            "date", "instrument", "open", "high", "low", "close", "volume", "amount",
            "amount_ma5", "volume_ma5", "amount_ma20", "volume_ma20",
            "close_ma20", "label", "label_cs", "in_pool",
        }
        candidates = [c for c in panel.columns if c not in exclude]
        scored = [(feature_quality(panel, c), c) for c in candidates]
        scored = [(s, c) for s, c in scored if s > EPS]
        scored.sort(reverse=True)
        core = [c for c in CORE_FEATURES if c in candidates]
        selected = list(dict.fromkeys(core + [c for _, c in scored]))[:MAX_FEATURES]
        if not selected:
            selected = [c for c in CORE_FEATURES if c in panel.columns]
        if not selected:
            raise RuntimeError("没有可用输入特征")
        logger.info("特征选择完成", n_features=len(selected), first_features=selected[:12])
        return selected

    def preprocess(panel, feature_cols, with_label):
        df = panel.copy()
        for c in feature_cols:
            if c not in df.columns:
                df[c] = 0.0
            df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[c] = df.groupby("date")[c].transform(cs_clean)
        if with_label:
            df["label"] = pd.to_numeric(df["label"], errors="coerce").replace([np.inf, -np.inf], np.nan)
            lo, hi = df["label"].quantile([0.001, 0.999])
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                df["label"] = df["label"].clip(lo, hi)
            df["label_cs"] = df.groupby("date")["label"].transform(cs_clean)
        return df.sort_values(["instrument", "date"]).reset_index(drop=True)

    def collect_sequences(panel, sd, ed, feature_cols, need_label, max_samples, left_pad, seed_offset):
        sd, ed = norm_day(sd), norm_day(ed)
        rng = np.random.default_rng(SEED + seed_offset)
        xs, ys, meta = [], [], []
        seen = 0
        for _, g in panel.sort_values("date").groupby("instrument", sort=False):
            values = g[feature_cols].to_numpy(dtype=np.float32, copy=True)
            values = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0)
            dates = pd.to_datetime(g["date"]).to_numpy()
            insts = g["instrument"].astype(str).to_numpy()
            in_pool = g["in_pool"].astype(bool).to_numpy()
            labels = g["label_cs"].to_numpy(dtype=np.float32, copy=True) if need_label else None
            for i in range(len(g)):
                d = pd.Timestamp(dates[i]).normalize()
                if d < sd or d > ed or not in_pool[i]:
                    continue
                if i + 1 < SEQ_LEN:
                    if not left_pad:
                        continue
                    pad = np.repeat(values[:1], SEQ_LEN - i - 1, axis=0)
                    seq = np.vstack([pad, values[: i + 1]])
                else:
                    seq = values[i - SEQ_LEN + 1: i + 1]
                y = 0.0
                if need_label:
                    y = labels[i]
                    if not np.isfinite(y):
                        continue
                item = (d, insts[i])
                if max_samples is None or len(xs) < max_samples:
                    xs.append(seq.astype(np.float32, copy=True))
                    ys.append(float(y))
                    meta.append(item)
                else:
                    seen += 1
                    j = rng.integers(0, seen + len(xs))
                    if j < max_samples:
                        xs[j] = seq.astype(np.float32, copy=True)
                        ys[j] = float(y)
                        meta[j] = item
        if not xs:
            return None, None, []
        return np.stack(xs).astype(np.float32), np.asarray(ys, dtype=np.float32), meta

    # ---------- PatchTST：patching + channel-independence ----------
    class PatchTST(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            d_model = 64
            n_patches = math.floor((SEQ_LEN + STRIDE - PATCH_LEN) / STRIDE) + 1
            self.n_features = n_features
            self.n_patches = n_patches
            self.patch_proj = nn.Linear(PATCH_LEN, d_model)
            self.pos = nn.Parameter(torch.zeros(1, n_patches, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=4,
                dim_feedforward=128,
                dropout=0.1,
                batch_first=True,
                activation="gelu",
                norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, num_layers=2)
            self.channel_gate = nn.Linear(d_model, 1)
            self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 32), nn.GELU(), nn.Linear(32, 1))

        def forward(self, x):
            mean = x.mean(dim=1, keepdim=True)
            std = x.std(dim=1, keepdim=True, unbiased=False).clamp_min(1e-5)
            x = (x - mean) / std
            x = x.permute(0, 2, 1)
            pad = x[:, :, -1:].expand(-1, -1, STRIDE)
            x = torch.cat([x, pad], dim=-1)
            patches = x.unfold(dimension=-1, size=PATCH_LEN, step=STRIDE)
            bsz, n_feat, n_patch, _ = patches.shape
            patches = patches.contiguous().view(bsz * n_feat, n_patch, PATCH_LEN)
            z = self.patch_proj(patches) + self.pos[:, :n_patch]
            z = self.encoder(z).mean(dim=1).view(bsz, n_feat, -1)
            w = torch.softmax(self.channel_gate(z).squeeze(-1), dim=1)
            pooled = (z * w.unsqueeze(-1)).sum(dim=1)
            return self.head(pooled).squeeze(-1)

    def fallback_score(panel, sd, ed, mode="blend"):
        df = panel[
            (panel["date"] >= norm_day(sd))
            & (panel["date"] <= norm_day(ed))
            & panel["in_pool"].astype(bool)
        ].copy()
        if df.empty:
            return pd.DataFrame(columns=["date", "instrument", "score"])
        weight_sets = {
            "blend": {
                "ret_1": -0.08, "ret_5": -0.10, "ret_10": -0.12, "ret_20": -0.18,
                "intraday_ret": -0.12, "close_to_ma20": -0.10,
                "range": -0.12, "body_ratio": -0.04, "close_location": -0.04,
                "shadow_balance": 0.04, "volatility_20": -0.16,
                "amount_ratio5": 0.06, "volume_ratio5": 0.04,
                "amount_ratio20": 0.12, "volume_ratio20": 0.08,
                "hf5m_ret": -0.06, "hf15m_ret": -0.06,
                "hf5m_range": -0.06, "hf15m_range": -0.06,
                "hf5m_amount_log": 0.05, "hf15m_amount_log": 0.04,
                "hf5m_spread": -0.08, "hf15m_spread": -0.08,
                "hf5m_imbalance": 0.12, "hf15m_imbalance": 0.10,
            },
            "reversal": {
                "ret_1": -0.16, "ret_5": -0.18, "ret_10": -0.18, "ret_20": -0.20,
                "ret_60": -0.08, "intraday_ret": -0.14, "close_to_ma20": -0.16,
                "overnight_gap": -0.06, "close_location": -0.06,
            },
            "micro": {
                "hf5m_imbalance": 0.22, "hf15m_imbalance": 0.18,
                "hf5m_spread": -0.16, "hf15m_spread": -0.14,
                "hf5m_ret": -0.08, "hf15m_ret": -0.08,
                "hf5m_amount_log": 0.08, "hf15m_amount_log": 0.06,
            },
            "micro2": {
                "hf5m_imbalance": 0.28, "hf15m_imbalance": 0.10,
                "hf5m_spread": -0.20, "hf15m_spread": -0.06,
                "hf5m_range": -0.08, "hf15m_range": -0.04,
                "hf5m_amount_log": 0.10,
            },
            "rev_micro": {
                "ret_1": -0.10, "ret_5": -0.10, "intraday_ret": -0.08,
                "hf5m_ret": -0.10, "hf15m_ret": -0.08,
                "hf5m_imbalance": 0.16, "hf15m_imbalance": 0.12,
                "hf5m_spread": -0.12,
            },
            "liquidity": {
                "amount_log": 0.08, "volume_log": 0.06,
                "amount_ratio5": 0.10, "volume_ratio5": 0.08,
                "amount_ratio20": 0.20, "volume_ratio20": 0.14,
                "hf5m_amount_log": 0.10, "hf15m_amount_log": 0.08,
                "hf5m_spread": -0.12, "hf15m_spread": -0.10,
            },
            "lowvol": {
                "range": -0.18, "hf5m_range": -0.12, "hf15m_range": -0.12,
                "volatility_5": -0.14, "volatility_10": -0.16, "volatility_20": -0.22,
                "body_ratio": -0.06, "upper_shadow": -0.06, "lower_shadow": 0.04,
            },
            "trend": {
                "ret_5": 0.10, "ret_10": 0.12, "ret_20": 0.16, "ret_40": 0.14, "ret_60": 0.12,
                "close_to_ma20": 0.12, "amount_ratio20": 0.08,
                "volatility_20": -0.08,
            },
        }
        weights = weight_sets.get(mode, weight_sets["blend"])
        df["score"] = 0.0
        used = 0
        for c, w in weights.items():
            if c in df.columns:
                df[c] = df.groupby("date")[c].transform(cs_clean)
                df["score"] += w * df[c].fillna(0.0)
                used += 1
        if used == 0:
            df["score"] = 0.0
        return df[["date", "instrument", "score"]]

    def mean_daily_ic(meta, y, pred):
        df = pd.DataFrame(meta, columns=["date", "instrument"])
        df["label"] = y
        df["pred"] = pred
        ics = []
        for _, g in df.groupby("date"):
            if len(g) < 30:
                continue
            x = pd.Series(g["pred"]).rank(pct=True)
            yy = pd.Series(g["label"]).rank(pct=True)
            if x.std(ddof=0) <= EPS or yy.std(ddof=0) <= EPS:
                continue
            ic = x.corr(yy)
            if np.isfinite(ic):
                ics.append(float(ic))
        return float(np.mean(ics)) if ics else 0.0

    def train_predict(hist_panel, pred_panel, feature_cols):
        def meta_frame(meta):
            return pd.DataFrame(meta, columns=["date", "instrument"])

        def align_score(meta, score_df):
            key = meta_frame(meta)
            if score_df is None or score_df.empty:
                return np.zeros(len(key), dtype=np.float64)
            aligned = key.merge(score_df[["date", "instrument", "score"]], how="left", on=["date", "instrument"])
            return pd.to_numeric(aligned["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float64)

        def cs_array(meta, values):
            df = meta_frame(meta)
            df["score"] = np.asarray(values, dtype=np.float64)
            df["score"] = df.groupby("date")["score"].transform(cs_clean)
            return df["score"].fillna(0.0).to_numpy(dtype=np.float64)

        def rows_for_meta(panel, meta, cols_use=None):
            key = meta_frame(meta)
            cols_use = feature_cols if cols_use is None else cols_use
            cols = ["date", "instrument"] + cols_use
            out = key.merge(panel[cols], how="left", on=["date", "instrument"])
            x = out[cols_use].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            return np.nan_to_num(x.to_numpy(dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)

        def fit_ridge(meta_valid, meta_pred, y_valid, train_start_value, train_end_value, cols_use, seed_offset):
            train_df = hist_panel[
                (hist_panel["date"] >= norm_day(train_start_value))
                & (hist_panel["date"] <= norm_day(train_end_value))
                & hist_panel["in_pool"].astype(bool)
                & hist_panel["label_cs"].notna()
            ].copy()
            if len(train_df) < 12000 or not cols_use:
                return None, None, -999.0
            if len(train_df) > 260000:
                train_df = train_df.sample(n=260000, random_state=SEED + seed_offset)
            x_train = train_df[cols_use].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
            x_train = np.nan_to_num(x_train.to_numpy(dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
            y_train = pd.to_numeric(train_df["label_cs"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float64)
            x_valid = rows_for_meta(hist_panel, meta_valid, cols_use)
            x_pred = rows_for_meta(pred_panel, meta_pred, cols_use)
            xtx = x_train.T @ x_train
            xty = x_train.T @ y_train
            eye = np.eye(x_train.shape[1], dtype=np.float64)
            best_valid, best_pred, best_ic = None, None, -999.0
            for alpha in (0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0):
                try:
                    coef = np.linalg.solve(xtx + alpha * eye, xty)
                except Exception:
                    continue
                v = x_valid @ coef
                p = x_pred @ coef
                for direction in (1.0, -1.0):
                    vv = cs_array(meta_valid, direction * v)
                    ic = mean_daily_ic(meta_valid, y_valid, vv)
                    if ic > best_ic:
                        best_ic = ic
                        best_valid = vv
                        best_pred = cs_array(meta_pred, direction * p)
            return best_valid, best_pred, best_ic

        xtr, ytr, _ = collect_sequences(
            hist_panel, TRAIN_START, TRAIN_END, feature_cols, True, MAX_TRAIN_SAMPLES, False, 0
        )
        xva, yva, meta_va = collect_sequences(
            hist_panel, VALID_START, VALID_END, feature_cols, True, MAX_VALID_SAMPLES, False, 1
        )
        if xtr is None or xva is None or len(xtr) < 2000 or len(xva) < 500:
            logger.warning("PatchTST 样本不足，使用规则兜底")
            return None

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if device.type == "cpu":
            torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
        model = PatchTST(len(feature_cols)).to(device)
        logger.info("PatchTST 初始化完成", device=str(device), n_features=len(feature_cols))
        loader = DataLoader(
            TensorDataset(torch.from_numpy(xtr), torch.from_numpy(ytr)),
            batch_size=BATCH,
            shuffle=True,
            drop_last=False,
        )
        valid_loader = DataLoader(
            TensorDataset(torch.from_numpy(xva), torch.from_numpy(yva)),
            batch_size=BATCH * 2,
            shuffle=False,
            drop_last=False,
        )
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        loss_fn = nn.SmoothL1Loss(beta=0.5)
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_loss, stale = float("inf"), 0
        for ep in range(EPOCHS):
            t0 = time.time()
            model.train()
            train_loss, nb = 0.0, 0
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
                train_loss += float(loss.detach().cpu())
                nb += 1
            model.eval()
            losses = []
            with torch.no_grad():
                for xb, yb in valid_loader:
                    xb = xb.to(device, non_blocking=True)
                    yb = yb.to(device, non_blocking=True)
                    losses.append(float(loss_fn(model(xb), yb).detach().cpu()))
            valid_loss = float(np.mean(losses))
            logger.info(
                "epoch 完成",
                epoch=ep + 1,
                train_loss=round(train_loss / max(nb, 1), 6),
                valid_loss=round(valid_loss, 6),
                elapsed=round(time.time() - t0, 2),
            )
            if valid_loss < best_loss - 1e-5:
                best_loss = valid_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale = 0
            else:
                stale += 1
                if stale >= 2:
                    break
        model.load_state_dict(best_state)
        model.eval()

        valid_pred = []
        with torch.no_grad():
            for i in range(0, len(xva), 4096):
                valid_pred.append(model(torch.from_numpy(xva[i: i + 4096]).to(device)).cpu().numpy())
        valid_pred = np.concatenate(valid_pred)
        ic_pos = mean_daily_ic(meta_va, yva, valid_pred)
        ic_neg = mean_daily_ic(meta_va, yva, -valid_pred)
        direction = 1.0 if ic_pos >= ic_neg else -1.0
        patch_valid = cs_array(meta_va, direction * valid_pred)
        patch_ic = mean_daily_ic(meta_va, yva, patch_valid)
        logger.info("验证集 PatchTST 方向", ic_pos=round(ic_pos, 5), ic_neg=round(ic_neg, 5), patch_ic=round(patch_ic, 5))

        xte, _, meta_te = collect_sequences(
            pred_panel, start_date, end_date, feature_cols, False, None, True, 2
        )
        if xte is None:
            return None
        patch_pred = []
        with torch.no_grad():
            for i in range(0, len(xte), 4096):
                pred = model(torch.from_numpy(xte[i: i + 4096]).to(device)).cpu().numpy()
                patch_pred.append(direction * pred)
        patch_pred = cs_array(meta_te, np.concatenate(patch_pred))

        def directional_component(name, valid_values, pred_values):
            valid_pos = cs_array(meta_va, valid_values)
            pred_pos = cs_array(meta_te, pred_values)
            ic_pos_local = mean_daily_ic(meta_va, yva, valid_pos)
            valid_neg = -valid_pos
            pred_neg = -pred_pos
            ic_neg_local = mean_daily_ic(meta_va, yva, valid_neg)
            if ic_neg_local > ic_pos_local:
                return name + "_neg", valid_neg, pred_neg, ic_neg_local
            return name, valid_pos, pred_pos, ic_pos_local

        components = [("patch", patch_valid, patch_pred, patch_ic)]

        for mode in ("blend", "reversal", "micro", "micro2", "rev_micro", "liquidity", "lowvol", "trend"):
            valid_raw = align_score(meta_va, fallback_score(hist_panel, VALID_START, VALID_END, mode=mode))
            pred_raw = align_score(meta_te, fallback_score(pred_panel, start_date, end_date, mode=mode))
            comp = directional_component("rule_" + mode, valid_raw, pred_raw)
            components.append(comp)

        core_cols = [
            c for c in feature_cols
            if not (c.startswith("fin_") or c.startswith("flib_") or c.startswith("exp_"))
        ]
        hf_cols = [c for c in feature_cols if c in CORE_FEATURES or c.startswith("hf5m_") or c.startswith("hf15m_")]
        ridge_specs = [
            ("ridge_full", TRAIN_START, TRAIN_END, feature_cols, 11),
            ("ridge_recent", "2022-01-01 00:00:00", TRAIN_END, feature_cols, 12),
            ("ridge_2023", "2023-01-01 00:00:00", TRAIN_END, feature_cols, 13),
            ("ridge_core", "2022-01-01 00:00:00", TRAIN_END, core_cols, 14),
            ("ridge_hf", "2022-01-01 00:00:00", TRAIN_END, hf_cols, 15),
        ]
        ridge_logs = {}
        for ridge_name, ridge_start, ridge_end, ridge_cols, seed_offset in ridge_specs:
            rv, rp, ric = fit_ridge(meta_va, meta_te, yva, ridge_start, ridge_end, ridge_cols, seed_offset)
            ridge_logs[ridge_name] = ric
            if rv is not None and rp is not None:
                components.append((ridge_name, rv, rp, ric))

        best_name = "patch"
        best_valid = patch_valid
        best_pred = patch_pred
        best_ic = patch_ic

        def consider(name, valid_values, pred_values):
            nonlocal best_name, best_valid, best_pred, best_ic
            valid_values = cs_array(meta_va, valid_values)
            pred_values = cs_array(meta_te, pred_values)
            ic = mean_daily_ic(meta_va, yva, valid_values)
            if ic > best_ic + 0.0003:
                best_name = name
                best_valid = valid_values
                best_pred = pred_values
                best_ic = ic

        for name, valid_values, pred_values, _ in components:
            consider(name, valid_values, pred_values)

        top_components = sorted(components, key=lambda item: item[3], reverse=True)[:7]
        for i in range(len(top_components)):
            for j in range(i + 1, len(top_components)):
                name_i, valid_i, pred_i, _ = top_components[i]
                name_j, valid_j, pred_j, _ = top_components[j]
                for w in (0.15, 0.25, 0.35, 0.45, 0.55, 0.65, 0.75, 0.85):
                    consider(
                        f"{name_i}_{w:.2f}+{name_j}_{1.0-w:.2f}",
                        w * valid_i + (1.0 - w) * valid_j,
                        w * pred_i + (1.0 - w) * pred_j,
                    )

        top3_pool = top_components[:5]
        tri_weights = [
            (0.50, 0.30, 0.20),
            (0.45, 0.35, 0.20),
            (0.40, 0.40, 0.20),
            (0.60, 0.25, 0.15),
            (0.34, 0.33, 0.33),
        ]
        for i in range(len(top3_pool)):
            for j in range(i + 1, len(top3_pool)):
                for k in range(j + 1, len(top3_pool)):
                    name_i, valid_i, pred_i, _ = top3_pool[i]
                    name_j, valid_j, pred_j, _ = top3_pool[j]
                    name_k, valid_k, pred_k, _ = top3_pool[k]
                    for w1, w2, w3 in tri_weights:
                        consider(
                            f"{name_i}_{w1:.2f}+{name_j}_{w2:.2f}+{name_k}_{w3:.2f}",
                            w1 * valid_i + w2 * valid_j + w3 * valid_k,
                            w1 * pred_i + w2 * pred_j + w3 * pred_k,
                        )

        anchor_name = best_name
        anchor_valid = best_valid.copy()
        anchor_pred = best_pred.copy()
        anchor_ic = best_ic
        best_objective = anchor_ic
        best_corr_to_anchor = 1.0

        def daily_abs_corr(meta, a, b):
            df = meta_frame(meta)
            df["a"] = np.asarray(a, dtype=np.float64)
            df["b"] = np.asarray(b, dtype=np.float64)
            corrs = []
            for _, g in df.groupby("date"):
                if len(g) < 30:
                    continue
                x = pd.to_numeric(g["a"], errors="coerce")
                y = pd.to_numeric(g["b"], errors="coerce")
                mask = x.notna() & y.notna()
                if mask.sum() < 30:
                    continue
                x = x[mask].rank(pct=True)
                y = y[mask].rank(pct=True)
                if x.std(ddof=0) <= EPS or y.std(ddof=0) <= EPS:
                    continue
                corr = x.corr(y)
                if np.isfinite(corr):
                    corrs.append(abs(float(corr)))
            return float(np.mean(corrs)) if corrs else 1.0

        def residualize(meta, values, anchor_values):
            df = meta_frame(meta)
            df["y"] = np.asarray(values, dtype=np.float64)
            df["x"] = np.asarray(anchor_values, dtype=np.float64)
            resid = np.zeros(len(df), dtype=np.float64)
            for _, idx in df.groupby("date").groups.items():
                idx_list = list(idx)
                y = pd.to_numeric(df.loc[idx_list, "y"], errors="coerce").replace([np.inf, -np.inf], np.nan)
                x = pd.to_numeric(df.loc[idx_list, "x"], errors="coerce").replace([np.inf, -np.inf], np.nan)
                y = cs_clean(y).to_numpy(dtype=np.float64)
                x = cs_clean(x).to_numpy(dtype=np.float64)
                denom = float(np.dot(x, x))
                beta = float(np.dot(x, y) / denom) if denom > EPS else 0.0
                resid[np.asarray(idx_list, dtype=int)] = y - beta * x
            return cs_array(meta, resid)

        def consider_hybrid(name, valid_values, pred_values):
            nonlocal best_name, best_valid, best_pred, best_ic, best_objective, best_corr_to_anchor
            valid_values = cs_array(meta_va, valid_values)
            pred_values = cs_array(meta_te, pred_values)
            ic = mean_daily_ic(meta_va, yva, valid_values)
            corr = daily_abs_corr(meta_va, valid_values, anchor_valid)
            # v7 比 v5 更保守：主干预测力优先，只给低相关扰动一点边际奖励。
            objective = ic - 0.035 * max(corr - 0.78, 0.0) + 0.014 * max(0.90 - corr, 0.0)
            if objective > best_objective + 0.00015 and ic > anchor_ic - 0.0025:
                best_name = name
                best_valid = valid_values
                best_pred = pred_values
                best_ic = ic
                best_objective = objective
                best_corr_to_anchor = corr

        residual_pool = top_components[:8]
        for name, valid_values, pred_values, _ in residual_pool:
            rv = residualize(meta_va, valid_values, anchor_valid)
            rp = residualize(meta_te, pred_values, anchor_pred)
            for resid_w in (0.04, 0.06, 0.08, 0.10, 0.12, 0.14, 0.16, 0.18, 0.22, 0.26, 0.30):
                consider_hybrid(
                    f"hybrid_anchor_{1.0-resid_w:.2f}+resid_{name}_{resid_w:.2f}",
                    (1.0 - resid_w) * anchor_valid + resid_w * rv,
                    (1.0 - resid_w) * anchor_pred + resid_w * rp,
                )

        for i in range(len(residual_pool)):
            for j in range(i + 1, len(residual_pool)):
                name_i, valid_i, pred_i, _ = residual_pool[i]
                name_j, valid_j, pred_j, _ = residual_pool[j]
                for mix_w in (0.45, 0.55, 0.65):
                    rv = residualize(meta_va, mix_w * valid_i + (1.0 - mix_w) * valid_j, anchor_valid)
                    rp = residualize(meta_te, mix_w * pred_i + (1.0 - mix_w) * pred_j, anchor_pred)
                    for resid_w in (0.06, 0.10, 0.14, 0.18, 0.24):
                        consider_hybrid(
                            f"hybrid_anchor_{1.0-resid_w:.2f}+resid_{name_i}_{mix_w:.2f}_{name_j}_{resid_w:.2f}",
                            (1.0 - resid_w) * anchor_valid + resid_w * rv,
                            (1.0 - resid_w) * anchor_pred + resid_w * rp,
                        )

        component_summary = ",".join(
            [f"{name}:{ic:.4f}" for name, _, _, ic in sorted(components, key=lambda item: item[3], reverse=True)[:8]]
        )
        logger.info(
            "验证集融合选择",
            best=best_name,
            best_ic=round(best_ic, 5),
            patch_ic=round(patch_ic, 5),
            anchor=anchor_name,
            anchor_ic=round(anchor_ic, 5),
            corr_to_anchor=round(best_corr_to_anchor, 5),
            objective=round(best_objective, 5),
            top_components=component_summary,
            ridge_logs={k: round(v, 5) for k, v in ridge_logs.items()},
        )
        out = meta_frame(meta_te)
        out["score"] = best_pred
        return out

    logger.info("开始构建训练数据", train_start=TRAIN_START, train_end=VALID_END)
    hist_panel_raw, _ = build_panel(DEV_TABLES, TRAIN_START, VALID_END, with_label=True, include_optional=True)
    feature_cols = choose_features(hist_panel_raw)
    hist_panel = preprocess(hist_panel_raw, feature_cols, with_label=True)

    logger.info("开始构建推理数据", start=str(start_date), end=str(end_date))
    pred_panel_raw, pred_pool = build_panel(PRED_TABLES, start_date, end_date, with_label=False, include_optional=True)
    pred_panel = preprocess(pred_panel_raw, feature_cols, with_label=False)

    try:
        score = train_predict(hist_panel, pred_panel, feature_cols)
    except Exception as exc:
        logger.warning("PatchTST 训练/预测失败，使用规则兜底", error=str(exc))
        score = None
    if score is None or score.empty:
        score = fallback_score(pred_panel, start_date, end_date)

    target_pool = pred_pool[
        (pred_pool["date"] >= norm_day(start_date)) & (pred_pool["date"] <= norm_day(end_date))
    ].drop_duplicates(["date", "instrument"])
    if target_pool.empty:
        target_pool = pred_panel[
            (pred_panel["date"] >= norm_day(start_date))
            & (pred_panel["date"] <= norm_day(end_date))
            & pred_panel["in_pool"].astype(bool)
        ][["date", "instrument"]].drop_duplicates()
    result = target_pool.merge(score, how="left", on=["date", "instrument"])
    result["score"] = pd.to_numeric(result["score"], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)

    exposure, exposure_cols = read_exposure(PRED_TABLES["exposure"], start_date, end_date)
    result = neutralize_score(result, exposure, exposure_cols)
    result = final_score(result)
    logger.info(
        "分数构建完成",
        rows=len(result),
        days=result["date"].nunique() if not result.empty else 0,
        instruments=result["instrument"].nunique() if not result.empty else 0,
    )
    return result[["date", "instrument", "score"]]


if __name__ == "__main__":
    from bigmodule import M
    import structlog

    logger = structlog.get_logger()
    datasources = {
        "instruments": "bigalpha_2026_instruments",
        "bar1d": "bigalpha_2026_bar1d",
        "bar5m": "bigalpha_2026_stock_bar5m",
        "bar15m": "bigalpha_2026_stock_bar15m",
        "financial": "bigalpha_2026_financial",
        "factorlib": "bigalpha_2026_factorlib",
        "exposure": "bigalpha_2026_exposure",
    }
    start_date, end_date = "2024-01-01 00:00:00", "2024-12-31 23:59:59"
    logger.info("计算分数", start=start_date, end=end_date)
    score_data = main(datasources, start_date, end_date)
    print(score_data.head())
    logger.info("开始评估分数")
    result = M.bigalpha_eval._latest(
        factor_data=score_data,
        show=True,
    )
